<table width="100%" bgcolor="#f4f5f6" style="width: 100%; background-color: #f4f5f6; border-left: 6px solid #ff9900; border-radius: 8px; border-collapse: collapse;">
  <tr>
    <td width="110" align="center" valign="middle" style="padding: 15px;">
      <img src="https://github.com/sansalamanca/shining-path_DP700/raw/main/data/logo%20libro.jpg" width="85" style="border-radius: 8px; display: block;">
    </td>
    <td valign="middle" style="padding: 15px 20px 15px 5px; font-family: Arial, sans-serif;">
      <p style="margin: 0 0 4px 0; font-size: 13px; color: #555555;">Excerpt from the Amazon book</p>
      <h2 style="margin: 4px 0 10px 0; font-size: 20px; font-weight: 900; color: #111111; border: none; padding: 0;">THE SHINING PATH TO DATA ENGINEER CERTIFICATE</h2>
      <p style="margin: 0; font-size: 14px; font-weight: bold;">
        <a href="https://www.amazon.com" target="_blank" style="color: #b12704; text-decoration: underline;">🛒 Buy on Amazon &rarr;</a>
      </p>
    </td>
  </tr>
</table>



In [4]:
# 1. Install libraries needed for our session
!pip install -q --no-warn-conflicts pyspark==3.5.3 delta-spark==3.2.1

# 2. Download repository, folders and files... sales.csv
!rm -rf shining-path_DP700
!git clone -q https://github.com/sansalamanca/shining-path_DP700.git

# 3. Initiate SparkSession with support for Delta Lake
from delta import configure_spark_with_delta_pip

builder = SparkSession.builder \
    .appName("Silver_Layer_Demo") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()
print("¡SparkSession and environment ready for the Silver layer!")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
¡SparkSession y entorno listos para la capa Silver!


In [5]:
from delta.tables import DeltaTable
from pyspark.sql.functions import col

silver_table_path = "shining-path_DP700/Lakehouse/Tables/fact_sales_silver"

# 1. Load data from Bronze / Files to seed our initial Silver table
df_initial_silver = spark.read.format("csv") \
    .option("header", "true") \
    .option("escape", '"') \
    .load("shining-path_DP700/Lakehouse/Files/sales.csv") \
    .limit(10)

# Save as initial Silver Delta table
df_initial_silver.write.format("delta").mode("overwrite").save(silver_table_path)
print("Initial Silver Delta table created successfully.")

Initial Silver Delta table created successfully.


In [6]:
from delta.tables import DeltaTable
from pyspark.sql.functions import col

# 1. Simulate reading incoming data from Bronze (e.g., loading first 15 rows which include updates and new rows)
df_bronze_raw = spark.read.format("csv") \
    .option("header", "true") \
    .option("escape", '"') \
    .load("shining-path_DP700/Lakehouse/Files/sales.csv") \
    .limit(15)

# 2. Clean data: Filter out nulls on primary keys, cast data types, and drop duplicates
df_bronze_clean = df_bronze_raw \
    .filter(col("SalesOrderNumber").isNotNull() & col("SalesOrderLineNumber").isNotNull()) \
    .withColumn("Quantity", col("Quantity").cast("integer")) \
    .withColumn("UnitPrice", col("UnitPrice").cast("decimal(10,2)")) \
    .withColumn("TaxAmount", col("TaxAmount").cast("decimal(10,2)")) \
    .dropDuplicates(["SalesOrderNumber", "SalesOrderLineNumber"])

# 3. Reference the target Silver Delta table by path
tabla_silver = DeltaTable.forPath(spark, silver_table_path)

# 4. Execute the MERGE (Upsert) using UpdateAll and InsertAll
tabla_silver.alias("target").merge(
    source = df_bronze_clean.alias("source"),
    condition = "target.SalesOrderNumber = source.SalesOrderNumber AND target.SalesOrderLineNumber = source.SalesOrderLineNumber"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

print("Bronze-to-Silver cleaning and MERGE (Upsert) completed successfully!")

Bronze-to-Silver cleaning and MERGE (Upsert) completed successfully!


In [9]:
# Read back the updated Silver Delta table
df_silver_result = spark.read.format("delta").load(silver_table_path)

print(f"Total rows in Silver table after Merge: {df_silver_result.count()}")
df_silver_result.select("SalesOrderNumber", "SalesOrderLineNumber", "CustomerName", "Item", "Quantity", "UnitPrice").show(15, truncate=False)

# in Fabric use:   display(df_silver_result)



Total rows in Silver table after Merge: 15
+----------------+--------------------+----------------+-----------------------+--------+---------+
|SalesOrderNumber|SalesOrderLineNumber|CustomerName    |Item                   |Quantity|UnitPrice|
+----------------+--------------------+----------------+-----------------------+--------+---------+
|SO43697         |1                   |Cole Watson     |Road-150 Red, 62       |1       |3578.27  |
|SO43698         |1                   |Rachael Martinez|Mountain-100 Silver, 44|1       |3399.99  |
|SO43699         |1                   |Sydney Wright   |Mountain-100 Silver, 44|1       |3399.99  |
|SO43700         |1                   |Ruben Prasad    |Road-650 Black, 62     |1       |699.10   |
|SO43701         |1                   |Christy Zhu     |Mountain-100 Silver, 44|1       |3399.99  |
|SO43702         |1                   |Colin Anand     |Road-150 Red, 44       |1       |3578.27  |
|SO43703         |1                   |Albert Alvarez  |R